# ERA5 天氣圖繪製 (2016-01-23 ~ 25)

**資料**：`data/era5/16jan23-24.nc`、`data/era5/16jan25.nc`（ERA5 pressure levels，0.25°，每小時，90–145°E、10–90°N）
**環境**：`Arakawa`（cartopy、metpy、ipywidgets、ffmpeg）

## 自動分析的判斷標準（初稿，需人工檢查）

**鋒面**（Thermal Front Parameter 法）
- 用 θe 判斷：地面鋒面取 925 hPa（`SFC_FRONT_LEVEL`），高空鋒面取該層；θe 先做平滑。
- 鋒面位在 θe 梯度最大的線上：TFP = −∇|∇θe|·n̂ = 0（n̂ 指向暖區）。
- 線段需同時符合：
  - |∇θe| > 4 K/100 km
  - 相對渦度 > 0（氣旋式風切）
  - 海平面氣壓 ∇²p > 0（位在低壓槽）
  - 100°E 以東
  - 長度 > 700 km
- 分類看垂直鋒面風速 v·n̂：
  - 大於 +1.5 m/s（冷空氣推向暖區）→ **冷鋒**
  - 小於 −1.5 m/s → **暖鋒**
  - 介於兩者之間 → **滯留鋒**
  - 距低壓中心 700 km 內且位在 θe 暖舌（∇²θe < 0）→ **囚錮鋒**

**槽線／脊線**（等高線曲率法）
- 高度場大幅平滑後算等高線曲率 κ = ∇·(∇z/|∇z|)，κ > 0 為氣旋式彎曲。
- 沿地轉風方向找 κ 的極值：
  - κ 極大且 > 門檻 → **槽**（等高線最低點連線；槽前偏南風、槽後偏北風）
  - κ 極小且 < −門檻 → **脊**（脊前偏北風、脊後偏南風）
- 只在西風帶分析：地轉風有向東分量、高度梯度不能太弱、緯度 15–70°N，且長度 > 600 km。

**H／L**
- 在平滑後的海平面氣壓中，用約 10°（40 格點）視窗找局部極大／極小值。
- 中心要比周圍平均高 ≥ 1.5 hPa（H）或低 ≥ 1 hPa（L）才算數。
- 中心氣壓四捨五入到最近的偶數。

**W／C**
- 在大幅平滑後的溫度場中，用約 15°（60 格點）視窗找局部極大（W）／極小（C）值。
- 中心要與周圍平均相差 ≥ 1.5 °C。

**限制**
- 高原上的氣壓是外插值，那裡的 H/L 不可靠。
- 鋒面門檻是用本個案（2016/1/23–25）調出來的，換其他個案可能需要調整。
- 截斷低壓等不在西風帶中的系統不畫槽線。

**使用方式**：依序執行所有 cell，最後一個 cell 會出現互動面板：
1. **高度層**：地面 Surface / 1000 / 925 / 850 / 700 / 500 / 300 / 250 / 200 hPa
2. **變數**：`完整天氣圖`（依規格畫全部元素）、單一元素，或 `自訂組合` 多選（Cmd/Ctrl + 點選）
3. **模式**：`單一時間 (PNG)` 或 `時段動畫 (MP4)`（選開始、結束、間隔與 FPS）

輸出檔存在 `output/`。

> **注意**
> - 資料檔**沒有海平面氣壓**，地面等壓線由 1000 hPa 重力位高度以壓高公式推估：$p_{msl}=1000\,\exp\!\left(\frac{g Z_{1000}}{R_d \bar T_v}\right)$。青藏高原等高地形處 1000 hPa 為外插值，等壓線僅供參考。若另外下載 ERA5 single-levels 的 `mean_sea_level_pressure` 並把檔案加入 `DATA_FILES`，程式會自動改用 `msl`。
> - **鋒面、槽脊線、H/L、W/C** 為客觀演算法自動分析的初稿（first guess），交作業前請依課堂規則人工檢查與修正。

In [ ]:
import os, shutil, time, warnings
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import matplotlib.patheffects as pe
from matplotlib.animation import FFMpegWriter
from matplotlib.figure import Figure
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.ndimage import gaussian_filter, maximum_filter, minimum_filter
import contourpy
from metpy.plots import ColdFront, WarmFront, StationaryFront, OccludedFront

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC', 'PingFang TC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
_ff = shutil.which('ffmpeg') or '/opt/homebrew/bin/ffmpeg'
plt.rcParams['animation.ffmpeg_path'] = _ff

## 1. 基本設定與讀取資料
可修改 `EXTENT`（繪圖範圍）、`PROJECTIONS`（可選投影，預設 `DEFAULT_PROJ` = 麥卡托）、`RES`（海岸線解析度，第一次用 10m 會自動下載 Natural Earth 資料）。

In [ ]:
# ============ 基本設定 ============
DATA_FILES = ['data/era5/16jan23-24.nc', 'data/era5/16jan25.nc']
OUT_DIR = 'output'
EXTENT = [90, 145, 10, 60]          # lon0, lon1, lat0, lat1 (資料範圍 90–145E, 10–90N)
PROJECTIONS = {                     # 互動面板「投影」下拉選單的選項
    '麥卡托 Mercator': ccrs.Mercator(central_longitude=117.5),
    '蘭伯特正形 Lambert Conformal': ccrs.LambertConformal(central_longitude=117.5, central_latitude=35,
                                                       standard_parallels=(25, 45)),
    '北極立體 Polar Stereographic': ccrs.NorthPolarStereo(central_longitude=117.5),
    '等距圓柱 PlateCarree': ccrs.PlateCarree(central_longitude=117.5),
}
DEFAULT_PROJ = '麥卡托 Mercator'
PC = ccrs.PlateCarree()
RES = '10m'                         # Natural Earth 高解析海岸線 (10m / 50m / 110m)
FRONT_LON_MIN = 100                 # 100°E 以西 (青藏高原/緬甸) 地形外插嚴重，不分析鋒面
SFC_FRONT_LEVEL = 925               # 地面鋒面以此層 θe 判斷

# 垂直剖面：A → B 直線 (位於 lat = 3.246010028067306 × lon − 369.3932899070966 上)
SECTION = dict(
    points=[(121.49318204166119, 24.974797341942413),   # A (lon, lat)
            (121.55353128360046, 25.17069158646357)],   # B (lon, lat)
    npts=161, top=200,                                  # 取樣點數、最高層 (hPa)
    terrain_file='data/terrain_section.npz',            # SRTM 30 m 地形快取 (沒有就自動下載)
)

G, RD, RE, OMEGA = 9.80665, 287.05, 6.371e6, 7.292e-5

os.makedirs(OUT_DIR, exist_ok=True)

# ============ 讀取資料 ============
ds = xr.open_mfdataset(DATA_FILES, combine='by_coords', chunks={'valid_time': 1},
                       data_vars='minimal', coords='minimal', compat='override')
ds = ds.sortby('latitude')                                   # 南 -> 北
ds = ds.sel(latitude=slice(EXTENT[2] - 2, EXTENT[3] + 2),
            longitude=slice(EXTENT[0], EXTENT[1]))
LAT = ds.latitude.values
LON = ds.longitude.values
LON2, LAT2 = np.meshgrid(LON, LAT)
TIMES = ds.valid_time.values
LEVELS = ds.pressure_level.values

print(ds)
print('時間:', TIMES[0], '~', TIMES[-1], f'({len(TIMES)} 筆)')

## 2. 數值工具
露點（Bolton 1980）、相當位溫 θe、海平面氣壓推估、極值（H/L、W/C）搜尋、等值線抽取、鋸齒線。

In [ ]:
# ============ 數值工具 ============
def smooth(a, sigma):
    return gaussian_filter(a, sigma, mode='nearest') if sigma else a


def ddx(f):
    return np.gradient(f, np.deg2rad(LON), axis=1) / (RE * np.cos(np.deg2rad(LAT2)))


def ddy(f):
    return np.gradient(f, np.deg2rad(LAT), axis=0) / RE


def get_level(t, p):
    """回傳某時間、某層的 z(gpm), T(°C), Td(°C), u, v (numpy arrays)"""
    s = ds.sel(valid_time=t, pressure_level=p).load()
    z = s.z.values / G
    T = s.t.values - 273.15
    q = np.clip(s.q.values, 1e-8, None)
    e = q * p / (0.622 + 0.378 * q)                          # hPa
    ln = np.log(e / 6.112)
    Td = 243.5 * ln / (17.67 - ln)                           # Bolton (1980)
    return dict(z=z, T=T, Td=Td, u=s.u.values, v=s.v.values, q=q, p=p)


def theta_e(T, Td, p):
    """Bolton (1980) 相當位溫 (K)"""
    Tk, Tdk = T + 273.15, Td + 273.15
    e = 6.112 * np.exp(17.67 * Td / (Td + 243.5))
    r = 0.622 * e / (p - e)
    Tl = 1 / (1 / (Tdk - 56) + np.log(Tk / Tdk) / 800) + 56
    th = Tk * (1000 / p) ** (0.2854 * (1 - 0.28 * r))
    return th * np.exp((3.376 / Tl - 0.00254) * r * 1e3 * (1 + 0.81 * r))


def get_mslp(t):
    """海平面氣壓 (hPa)。檔案若有 msl 直接用；否則用 1000 hPa 高度以壓高公式推估。"""
    if 'msl' in ds:
        return ds.msl.sel(valid_time=t).values / 100
    d = get_level(t, 1000)
    Tv = (d['T'] + 273.15) * (1 + 0.61 * d['q'])
    Tbar = Tv + 0.0065 * d['z'] / 2                         # 1000 hPa 與海平面間平均虛溫
    return 1000 * np.exp(G * d['z'] / (RD * Tbar))


def find_extrema(f, size, kind, min_dev, edge=8):
    """區域極值：size=搜尋窗(格點)，min_dev=與周圍平均差值門檻"""
    filt = maximum_filter if kind == 'max' else minimum_filter
    ext = (f == filt(f, size=size, mode='nearest'))
    mean = gaussian_filter(f, size / 2, mode='nearest')
    dev = (f - mean) if kind == 'max' else (mean - f)
    ext &= dev > min_dev
    ext[:edge, :] = ext[-edge:, :] = ext[:, :edge] = ext[:, -edge:] = False
    j, i = np.where(ext)
    return [(LON[ii], LAT[jj], f[jj, ii]) for jj, ii in zip(j, i)]


def zero_lines(field, mask=None, min_len_km=600):
    """field=0 的等值線段 (lon, lat)；mask 為 True 處不畫"""
    z = np.ma.masked_array(field, mask=mask if mask is not None else False)
    gen = contourpy.contour_generator(LON, LAT, z, line_type='Separate')
    out = []
    for seg in gen.lines(0.0):
        if len(seg) < 3:
            continue
        dl = np.hypot(np.diff(seg[:, 0]) * np.cos(np.deg2rad(seg[:-1, 1])), np.diff(seg[:, 1]))
        if dl.sum() * 111 >= min_len_km:
            out.append(seg)
    return out


def interp(f, lon, lat):
    i = np.clip(np.round((lon - LON[0]) / (LON[1] - LON[0])).astype(int), 0, len(LON) - 1)
    j = np.clip(np.round((lat - LAT[0]) / (LAT[1] - LAT[0])).astype(int), 0, len(LAT) - 1)
    return f[j, i]


def zigzag(seg, amp=0.4, wl=1.5):
    """把線段轉成鋸齒線 (單位: 度)"""
    x, y = seg[:, 0] * np.cos(np.deg2rad(seg[:, 1])), seg[:, 1]
    s = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    if s[-1] < wl:
        return seg
    n = int(s[-1] / (wl / 2))
    si = np.linspace(0, s[-1], n + 1)
    xi, yi = np.interp(si, s, x), np.interp(si, s, y)
    tx, ty = np.gradient(xi), np.gradient(yi)
    nn = np.hypot(tx, ty) + 1e-12
    sign = np.where(np.arange(n + 1) % 2 == 0, 1, -1)
    sign[0] = sign[-1] = 0
    xz, yz = xi - ty / nn * amp * sign, yi + tx / nn * amp * sign
    return np.c_[xz / np.cos(np.deg2rad(yz)), yz]

## 3. 槽線 / 脊線
以等高線曲率 κ = ∇·(∇z/|∇z|) 判斷：沿地轉風方向 κ 為極大（氣旋式）者為**槽**，極小（反氣旋式）者為**脊**；只在西風帶分析。
對應規則：槽前偏南風、槽後偏北風；脊前偏北風、脊後偏南風；槽線接近等高線最低點、脊線接近最高點。

In [ ]:
# ============ 槽、脊線 ============
def trough_ridge(z, sigma=8, kthr=1.5e-6, lat_rng=(15, 70), min_len_km=600):
    """
    西風帶中的槽/脊線 (以等高線曲率判斷)：
      n̂ = ∇z/|∇z|，等高線曲率 κ = ∇·n̂ (>0 氣旋式)，s = k×n̂ 為地轉風方向
      槽：∂κ/∂s = 0 且 κ 沿流向為極大、κ > kthr  → 槽前偏南風、槽後偏北風
      脊：∂κ/∂s = 0 且 κ 沿流向為極小、κ < -kthr → 脊前偏北風、脊後偏南風
    只保留西風帶 (地轉風有向東分量) 且高度梯度不太弱的區域
    """
    zs = smooth(z, sigma)
    zx, zy = ddx(zs), ddy(zs)
    gm = np.hypot(zx, zy) + 1e-12
    nx, ny = zx / gm, zy / gm
    kap = smooth(ddx(nx) + ddy(ny), 2)
    sx, sy = -ny, nx
    ks = sx * ddx(kap) + sy * ddy(kap)
    kss = sx * ddx(ks) + sy * ddy(ks)
    base = (sx < 0.2) | (gm < np.nanpercentile(gm, 25)) | (LAT2 < lat_rng[0]) | (LAT2 > lat_rng[1])
    tro = zero_lines(ks, base | (kss >= 0) | (kap < kthr), min_len_km)
    rid = zero_lines(ks, base | (kss <= 0) | (kap > -kthr), min_len_km)
    return tro, rid

## 4. 鋒面（Thermal Front Parameter）
- 地面鋒面用 `SFC_FRONT_LEVEL`（預設 925 hPa）θe；高空鋒面用該層 θe。
- 條件：θe 梯度大、相對渦度 > 0（氣旋式風切）、海平面氣壓 ∇²p > 0（位於低壓槽）。
- 分類：垂直鋒面風速由冷側吹向暖側 → 冷鋒；反之 → 暖鋒；很小 → 滯留鋒；靠近低壓中心且位於 θe 暖舌 → 囚錮鋒。
- 冷鋒 = 藍色三角形、暖鋒 = 紅色半圓、滯留鋒 = 紅半圓 + 藍三角形（兩側）、囚錮鋒 = 紫色半圓 + 三角形（同側）。

In [ ]:
# ============ 鋒面 (客觀分析: Thermal Front Parameter) ============
def detect_fronts(t, p_front, mslp=None, sigma=5, grad_thr=4.0, min_len_km=700):
    """
    τ = θe(p_front)。TFP = -∇|∇τ|·n̂，鋒面位於 TFP=0 且 |∇τ| 為極大值處，並要求：
      |∇τ| > grad_thr (K/100km)、相對渦度 > 0 (氣旋式風切)、
      海平面氣壓拉普拉斯 > 0 (位於低壓槽)。
    分類：垂直鋒面風速 v·n̂ (n̂ 指向暖區)
      v·n̂ > +1.5 → 冷鋒 (冷空氣推向暖區)；v·n̂ < -1.5 → 暖鋒；其餘 → 滯留鋒；
      靠近低壓中心且位於 θe 暖舌 (∇²τ<0) → 囚錮鋒
    """
    d = get_level(t, p_front)
    tau = smooth(theta_e(d['T'], d['Td'], p_front), sigma)
    tx, ty = ddx(tau), ddy(tau)
    gm = np.hypot(tx, ty)
    nx, ny = tx / (gm + 1e-12), ty / (gm + 1e-12)
    gx, gy = ddx(gm), ddy(gm)
    tfp = -(gx * nx + gy * ny)
    loc = ddx(tfp) * nx + ddy(tfp) * ny                     # >0 → |∇τ| 極大
    u, v = smooth(d['u'], sigma), smooth(d['v'], sigma)
    vort = ddx(v) - ddy(u) + u * np.tan(np.deg2rad(LAT2)) / RE
    lap_tau = ddx(tx) + ddy(ty)
    mask = (gm * 1e5 < grad_thr) | (loc <= 0) | (vort <= 0) | (LON2 < FRONT_LON_MIN)
    if mslp is not None:
        ps = smooth(mslp, sigma)
        mask |= (ddx(ddx(ps)) + ddy(ddy(ps))) <= 0
    segs = zero_lines(tfp, mask, min_len_km=min_len_km)
    vn = u * nx + v * ny
    lows = find_extrema(smooth(mslp, 4), 40, 'min', 1.0) if mslp is not None else []

    fronts = []
    for seg in segs:
        # 依鋒面類型切成小段
        vn_s = interp(vn, seg[:, 0], seg[:, 1])
        lt_s = interp(lap_tau, seg[:, 0], seg[:, 1])
        k = np.full(len(seg), 'stationary', dtype=object)
        k[vn_s > 1.5] = 'cold'
        k[vn_s < -1.5] = 'warm'
        for lo, la, _ in lows:
            dist = np.hypot((seg[:, 0] - lo) * np.cos(np.deg2rad(la)), seg[:, 1] - la) * 111
            k[(dist < 700) & (lt_s < -2e-10)] = 'occluded'
        # 平滑分類 (眾數濾波)
        kk = k.copy()
        w = 6
        for i in range(len(k)):
            vals, cnt = np.unique(k[max(0, i - w):i + w + 1], return_counts=True)
            kk[i] = vals[np.argmax(cnt)]
        brk = np.where(kk[1:] != kk[:-1])[0] + 1
        for a, b in zip(np.r_[0, brk], np.r_[brk, len(seg)]):
            piece = seg[max(a - 1, 0):b + 1]
            if len(piece) < 4:
                continue
            # 方向：讓「暖區」位於前進方向左側 (metpy 符號畫在左側)
            mx, my = piece[:, 0].mean(), piece[:, 1].mean()
            tvx = (piece[-1, 0] - piece[0, 0]) * np.cos(np.deg2rad(my))
            tvy = piece[-1, 1] - piece[0, 1]
            wnx, wny = interp(nx, np.array([mx]), np.array([my]))[0], interp(ny, np.array([mx]), np.array([my]))[0]
            if (-tvy * wnx + tvx * wny) < 0:
                piece = piece[::-1]
            fronts.append((kk[a], piece))
    return fronts


FRONT_STYLE = {
    'cold':       dict(color='blue', pe=lambda: ColdFront(size=6, color='blue')),
    'warm':       dict(color='red', pe=lambda: WarmFront(size=6, color='red', flip=True)),
    'stationary': dict(color='red', pe=lambda: StationaryFront(size=6, colors=('red', 'blue'))),
    'occluded':   dict(color='purple', pe=lambda: OccludedFront(size=6, color='purple')),
}


def draw_fronts(ax, fronts):
    for kind, seg in fronts:
        st = FRONT_STYLE[kind]
        ax.plot(seg[:, 0], seg[:, 1], color=st['color'], lw=2.2, transform=PC,
                path_effects=[st['pe']()], zorder=6)

## 5. 繪圖規格與函式
| 層 | 等高線/等壓線 | 標示 | 等溫線 | 標示 |
|---|---|---|---|---|
| Surface | 黑實線 4 hPa（…996, 1000, 1004…） | 每 20 hPa | – | – |
| 850 hPa | 黑實線 30 gpm（…1470, 1500, 1530…） | 1500 | 紅虛線 3 °C | −15, 0, +15 |
| 500 hPa | 黑實線 60 gpm（…5760, 5820, 5880…） | 5760, 5880 | 紅虛線 5 °C | −10, −40 |

其他層（1000/925/700/300/250/200）使用 `LEVEL_SPEC` 中的合理預設值，可自行修改。
潮濕區 T−Td ≤ 3 °C 綠色塗色；槽線棕色虛線、脊線棕色鋸齒線；H 藍、L 紅（標示偶數中心氣壓）；W 紅、C 藍（不標數值）。等值線皆先高斯平滑，使線條平滑且不分岔、不相交。

In [ ]:
# ============ 各層繪圖規格 ============
# hgt: (間距, 基準值, 需標示的值或 'every:N')；tmp: (間距, 需標示的值)
LEVEL_SPEC = {
    'Surface': dict(hgt=(4, 1000, 'every:20'), tmp=None),
    1000: dict(hgt=(30, 120, 'every:60'), tmp=(3, [-15, 0, 15])),
    925:  dict(hgt=(30, 780, 'every:120'), tmp=(3, [-15, 0, 15])),
    850:  dict(hgt=(30, 1500, [1500]), tmp=(3, [-15, 0, 15])),
    700:  dict(hgt=(30, 3000, [3000]), tmp=(3, [-15, 0])),
    500:  dict(hgt=(60, 5760, [5760, 5880]), tmp=(5, [-10, -40])),
    300:  dict(hgt=(120, 9120, [9120]), tmp=(5, [-40])),
    250:  dict(hgt=(120, 10320, [10320]), tmp=(5, [-50])),
    200:  dict(hgt=(120, 11760, [11760]), tmp=(5, [-50])),
}

ELEMENTS = {
    'Surface': ['等壓線 Isobars', '高低壓中心 H/L', '鋒面 Fronts', '風標 Wind'],
    'upper':   ['等高線 Height', '等溫線 Isotherms', '潮濕區 T-Td≤3°C', '槽線/脊線 Trough/Ridge',
                '冷暖中心 W/C', '鋒面 Fronts', '風標 Wind'],
    'Section': ['位溫 θ', '等溫線 Isotherms', '潮濕區 T-Td≤3°C', '風標 Wind',
                '相當位溫 θe', '法向風 Normal wind'],
}
DEFAULT_ON = {'風標 Wind': False, '鋒面 Fronts': None}  # None → 只有 Surface 預設開

BROWN = '#8B4513'


def contour_levels(f, step, base):
    lo = base + np.floor((np.nanmin(f) - base) / step) * step
    hi = base + np.ceil((np.nanmax(f) - base) / step) * step
    return np.arange(lo, hi + step / 2, step)


def label_list(levels, rule):
    if isinstance(rule, str) and rule.startswith('every:'):
        n = float(rule.split(':')[1])
        return [l for l in levels if np.isclose(l % n, 0)]
    return [l for l in levels if any(np.isclose(l, r) for r in rule)]


def add_clabels(ax, cs, lv, fmt):
    lv = [l for l in lv if l in cs.levels]
    if lv:
        tx = ax.clabel(cs, levels=lv, inline=True, fontsize=9, fmt=fmt, inline_spacing=3)
        for t_ in tx:
            t_.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])


def mark_centers(ax, pts, letter, color, show_val, fmt='{:.0f}'):
    for lo, la, val in pts:
        if not (EXTENT[0] + 1 < lo < EXTENT[1] - 1 and EXTENT[2] + 1 < la < EXTENT[3] - 1):
            continue
        ax.text(lo, la, letter, color=color, fontsize=20, fontweight='bold', ha='center',
                va='center', transform=PC, zorder=8,
                path_effects=[pe.withStroke(linewidth=3, foreground='white')])
        if show_val:
            ax.text(lo, la - 1.1, fmt.format(val), color=color, fontsize=10, fontweight='bold',
                    ha='center', va='top', transform=PC, zorder=8,
                    path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])


def setup_map(fig, proj=DEFAULT_PROJ):
    crs = PROJECTIONS[proj]
    ax = fig.add_subplot(1, 1, 1, projection=crs)
    ax.set_extent(EXTENT, PC)
    if not isinstance(crs, (ccrs.Mercator, ccrs.PlateCarree)):
        _fan_boundary(ax)
    return _decorate(ax)


def _fan_boundary(ax):
    """非圓柱投影：把地圖邊界裁成經緯度範圍 (扇形)，避免角落沒有資料"""
    n = 80
    lo0, lo1, la0, la1 = EXTENT
    verts = np.r_[np.c_[np.linspace(lo0, lo1, n), np.full(n, la0)],
                  np.c_[np.full(n, lo1), np.linspace(la0, la1, n)],
                  np.c_[np.linspace(lo1, lo0, n), np.full(n, la1)],
                  np.c_[np.full(n, lo0), np.linspace(la1, la0, n)]]
    xy = ax.projection.transform_points(PC, verts[:, 0], verts[:, 1])[:, :2]
    ax.set_boundary(mpath.Path(xy), transform=ax.transData)


def _decorate(ax):
    ax.add_feature(cfeature.LAND.with_scale(RES), facecolor='#f4f1e8', zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale(RES), facecolor='#eaf3f8', zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale(RES), lw=0.6, edgecolor='#555', zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale(RES), lw=0.4, edgecolor='#888', linestyle='-', zorder=2)
    ax.add_feature(cfeature.LAKES.with_scale(RES), facecolor='#eaf3f8', edgecolor='#777', lw=0.3, zorder=1)
    gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False, lw=0.4, color='gray',
                      alpha=0.6, linestyle=':')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = dict(size=9)
    return ax


def plot_chart(fig, t, level, elements, proj=DEFAULT_PROJ):
    """在 fig 上畫一張天氣圖。level='Surface' 或 hPa 數值；elements=要畫的元素 list"""
    fig.clf()
    if level == 'Section':
        return plot_section(fig, t, elements)
    ax = setup_map(fig, proj)
    spec = LEVEL_SPEC.get(level, dict(hgt=(60, 0, []), tmp=(5, [])))
    tstr = np.datetime_as_string(t, unit='h').replace('T', ' ') + ' UTC'
    has = lambda key: any(e.startswith(key) for e in elements)

    if level == 'Surface':
        mslp = get_mslp(t)
        ps = smooth(mslp, 4)
        step, base, rule = spec['hgt']
        if has('等壓線'):
            lv = contour_levels(ps, step, base)
            cs = ax.contour(LON, LAT, ps, levels=lv, colors='k', linewidths=1.0, transform=PC, zorder=4)
            add_clabels(ax, cs, label_list(lv, rule), '%d')
        if has('高低壓中心'):
            even = lambda pts: [(a, b, 2 * np.round(c / 2)) for a, b, c in pts]
            mark_centers(ax, even(find_extrema(ps, 40, 'max', 1.5)), 'H', 'blue', True)
            mark_centers(ax, even(find_extrema(ps, 40, 'min', 1.0)), 'L', 'red', True)
        if has('鋒面'):
            draw_fronts(ax, detect_fronts(t, SFC_FRONT_LEVEL, mslp=mslp))
        if has('風標'):
            d = get_level(t, 1000)
            draw_barbs(ax, d['u'], d['v'])
        title = f'Surface (MSLP est. from Z1000)  |  {tstr}'
    else:
        d = get_level(t, level)
        zs, Ts = smooth(d['z'], 3), smooth(d['T'], 3)
        dep = smooth(d['T'] - d['Td'], 2)
        if has('潮濕區'):
            ax.contourf(LON, LAT, dep, levels=[-50, 3], colors=['#2ca02c'], alpha=0.35,
                        transform=PC, zorder=3)
        if has('等高線'):
            step, base, rule = spec['hgt']
            lv = contour_levels(zs, step, base)
            cs = ax.contour(LON, LAT, zs, levels=lv, colors='k', linewidths=1.1, transform=PC, zorder=4)
            add_clabels(ax, cs, label_list(lv, rule), '%d')
        if has('等溫線') and spec['tmp']:
            step, lab = spec['tmp']
            lv = contour_levels(Ts, step, 0)
            cs = ax.contour(LON, LAT, Ts, levels=lv, colors='red', linewidths=1.0, linestyles='--',
                            transform=PC, zorder=4)
            add_clabels(ax, cs, label_list(lv, lab), lambda x: f'{x:+.0f}' if x else '0')
        if has('槽線'):
            tro, rid = trough_ridge(d['z'])
            for s in tro:
                ax.plot(s[:, 0], s[:, 1], color=BROWN, lw=2.4, ls=(0, (6, 3)), transform=PC, zorder=5)
            for s in rid:
                zz = zigzag(s)
                ax.plot(zz[:, 0], zz[:, 1], color=BROWN, lw=1.8, transform=PC, zorder=5)
        if has('冷暖中心'):
            Tc = smooth(d['T'], 6)
            mark_centers(ax, find_extrema(Tc, 60, 'max', 1.5), 'W', 'red', False)
            mark_centers(ax, find_extrema(Tc, 60, 'min', 1.5), 'C', 'blue', False)
        if has('鋒面'):
            draw_fronts(ax, detect_fronts(t, level, mslp=get_mslp(t)))
        if has('風標'):
            draw_barbs(ax, d['u'], d['v'])
        title = f'{level} hPa  |  {tstr}'
    ax.set_title(title, fontsize=13, loc='left')
    ax.set_title('ERA Reanalysis', fontsize=10, loc='right')
    ax.annotate('Credit: S.-H. Ho', xy=(1, 1), xycoords='axes fraction', xytext=(0, 22),
                textcoords='offset points', fontsize=7, color='#444', ha='right', va='bottom')
    return ax


def draw_barbs(ax, u, v, skip=10):
    sl = (slice(None, None, skip), slice(None, None, skip))
    ax.barbs(LON2[sl], LAT2[sl], u[sl] * 1.944, v[sl] * 1.944, length=5, lw=0.5,
             color='#333', transform=PC, zorder=5)


def figsize(level):
    return (12, 7) if level == 'Section' else (11, 9)


def save_png(t, level, elements, proj=DEFAULT_PROJ, dpi=600):
    fig = Figure(figsize=figsize(level))   # 不經 pyplot，避免 Jupyter 自動重複顯示
    plot_chart(fig, t, level, elements, proj)
    tag = {'Surface': 'sfc', 'Section': 'xsec'}.get(level, f'{level}hPa')
    fn = os.path.join(OUT_DIR, f'{tag}_{np.datetime_as_string(t, unit="h").replace(":", "")}.png')
    fig.savefig(fn, dpi=dpi, bbox_inches='tight')
    return fig, fn


def save_mp4(t0, t1, level, elements, proj=DEFAULT_PROJ, step_h=1, fps=4, dpi=600, progress=None):
    times = [t for t in TIMES if t0 <= t <= t1][::step_h]
    tag = {'Surface': 'sfc', 'Section': 'xsec'}.get(level, f'{level}hPa')
    s0 = np.datetime_as_string(t0, unit='h').replace(':', '')
    s1 = np.datetime_as_string(t1, unit='h').replace(':', '')
    fn = os.path.join(OUT_DIR, f'{tag}_{s0}_{s1}.mp4')
    fig = Figure(figsize=figsize(level))   # 不經 pyplot，避免 Jupyter 自動重複顯示
    writer = FFMpegWriter(fps=fps, bitrate=3000, extra_args=['-pix_fmt', 'yuv420p', '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2'])
    with writer.saving(fig, fn, dpi=dpi):
        for k, t in enumerate(times):
            plot_chart(fig, t, level, elements, proj)
            writer.grab_frame()
            if progress is not None:
                progress(k + 1, len(times))
    return fn

## 6. 垂直剖面 Cross-section
- 剖面線：A (24.9748°N, 121.4932°E) → B (25.1707°N, 121.5535°E)，位於 lat = 3.246010028067306 × lon − 369.3932899070966 上，長約 22.6 km（設定在 `SECTION`）。
- 地形：SRTM 30 m（OpenTopoData API），第一次執行時下載並存到 `data/terrain_section.npz`，之後直接讀檔；以深棕色填滿，蓋住地形以下的 ERA5 外插值。
- 橫軸：沿剖面距離（km）與經緯度座標；縱軸：重力位高度（gpm），由地面到 200 hPa；右軸標出各氣壓層的平均高度。
- 元素：
  - 位溫 θ：黑實線，每 2 K
  - 等溫線：紅虛線，每 5 °C
  - 潮濕區 T−Td ≤ 3 °C：綠色塗色
  - 風標：水平風，單位 kt，上為北、右為東
  - 相當位溫 θe：紫線，每 4 K
  - 法向風：垂直剖面的風速，+ 為吹向剖面右側（東南東）
- 注意：
  - ERA5 解析度為 0.25°（約 27 km），比這條 22.6 km 的剖面還粗：剖面上的氣象場是格點內插的結果，幾乎看不到水平變化，主要是看垂直結構。

In [ ]:
# ============ 垂直剖面 ============
def section_line():
    """A → B 取樣點 (距離 km, lon, lat) 與單位切向量 (東, 北)"""
    (lo0, la0), (lo1, la1) = SECTION['points']
    lon = np.linspace(lo0, lo1, SECTION['npts'])
    lat = np.linspace(la0, la1, SECTION['npts'])
    coslat = np.cos(np.deg2rad(lat.mean()))
    dx, dy = np.diff(lon) * coslat * 111.195, np.diff(lat) * 111.195
    dist = np.r_[0, np.cumsum(np.hypot(dx, dy))]
    L = np.hypot((lo1 - lo0) * coslat, la1 - la0)
    return dist, lon, lat, ((lo1 - lo0) * coslat / L, (la1 - la0) / L)


def get_terrain(lon, lat):
    """剖面上的地形高度 (m)：SRTM 30 m (OpenTopoData)，下載一次後存成快取"""
    f = SECTION['terrain_file']
    if os.path.exists(f):
        c = np.load(f)
        if c['lon'].shape == lon.shape and np.allclose(c['lon'], lon) and np.allclose(c['lat'], lat):
            return c['elev']
    import json, urllib.request
    elev = []
    for k in range(0, len(lon), 100):                    # API 每次最多 100 點、每秒 1 次
        loc = '|'.join(f'{a:.6f},{b:.6f}' for a, b in zip(lat[k:k + 100], lon[k:k + 100]))
        with urllib.request.urlopen(f'https://api.opentopodata.org/v1/srtm30m?locations={loc}',
                                    timeout=30) as r:
            elev += [p['elevation'] or 0.0 for p in json.load(r)['results']]
        time.sleep(1.1)
    elev = np.array(elev, dtype=float)
    np.savez(f, lon=lon, lat=lat, elev=elev)
    return elev


def get_section(t):
    dist, lon, lat, tang = section_line()
    sub = ds.sel(valid_time=t, pressure_level=slice(1000, SECTION['top']),
                 latitude=slice(lat.min() - 0.5, lat.max() + 0.5),
                 longitude=slice(lon.min() - 0.5, lon.max() + 0.5)).load()
    s = sub.interp(latitude=xr.DataArray(lat, dims='s'), longitude=xr.DataArray(lon, dims='s'))
    p = s.pressure_level.values[:, None] * np.ones((1, len(dist)))
    T = s.t.values - 273.15
    q = np.clip(s.q.values, 1e-8, None)
    e = q * p / (0.622 + 0.378 * q)
    ln = np.log(e / 6.112)
    Td = 243.5 * ln / (17.67 - ln)
    u, v = s.u.values, s.v.values
    try:
        terr = get_terrain(lon, lat)
    except Exception as err:
        print('地形下載失敗，剖面不畫地形:', err)
        terr = None
    return dict(dist=dist, lon=lon, lat=lat, p=p, terr=terr, z=s.z.values / G, T=T, Td=Td, u=u, v=v,
                th=(T + 273.15) * (1000 / p) ** 0.2854, the=theta_e(T, Td, p),
                vn=u * tang[1] - v * tang[0])          # 法向風：+ 為切向量右側


def plot_section(fig, t, elements):
    d = get_section(t)
    X = np.broadcast_to(d['dist'], d['z'].shape)
    Z = d['z']
    has = lambda key: any(e.startswith(key) for e in elements)
    ax = fig.add_axes([0.07, 0.20, 0.68, 0.70])
    if has('潮濕區'):
        ax.contourf(X, Z, d['T'] - d['Td'], levels=[-50, 3], colors=['#2ca02c'], alpha=0.35, zorder=1)
    if has('法向風'):
        lv = np.arange(5, 81, 5)
        cs = ax.contour(X, Z, d['vn'], levels=lv, colors='#d95f02', linewidths=0.9, zorder=2)
        ax.clabel(cs, fontsize=7, fmt='%d')
        cs = ax.contour(X, Z, d['vn'], levels=-lv[::-1], colors='#1b7837', linewidths=0.9,
                        linestyles='--', zorder=2)
        ax.clabel(cs, fontsize=7, fmt='%d')
    if has('相當位溫'):
        cs = ax.contour(X, Z, d['the'], levels=contour_levels(d['the'], 4, 300), colors='#7b2cbf',
                        linewidths=0.9, zorder=3)
        ax.clabel(cs, fontsize=7, fmt='%d')
    if has('位溫'):
        cs = ax.contour(X, Z, d['th'], levels=contour_levels(d['th'], 2, 300), colors='k',
                        linewidths=1.0, zorder=3)
        ax.clabel(cs, cs.levels[::2], fontsize=7, fmt='%d')
    if has('等溫線'):
        lv = contour_levels(d['T'], 5, 0)
        cs = ax.contour(X, Z, d['T'], levels=lv, colors='red', linewidths=1.0, linestyles='--', zorder=3)
        add_clabels(ax, cs, [l for l in lv if l % 10 == 0], lambda x: f'{x:+.0f}' if x else '0')
    if has('風標'):
        ix = np.linspace(0, len(d['dist']) - 1, 7).round().astype(int)
        barb_lev = [1000, 950, 900, 850, 800, 700, 600, 500, 400, 300, 250, 200]
        for k in np.where(np.isin(d['p'][:, 0], barb_lev))[0]:
            ax.barbs(X[k, ix], Z[k, ix], d['u'][k, ix] * 1.944, d['v'][k, ix] * 1.944,
                     length=5.5, lw=0.5, color='#333', zorder=4)

    if d['terr'] is not None:                         # 地形：灰色填滿，蓋住地形以下的外插值
        ax.fill_between(d['dist'], 0, d['terr'], color='#6b5b4b', zorder=5)
        ax.plot(d['dist'], d['terr'], color='k', lw=0.6, zorder=5)
    ax.set_xlim(0, d['dist'][-1])
    ax.set_ylim(0, Z[-1].max() + 200)
    ax.set_ylabel('Geopotential height (gpm)')
    ticks = np.linspace(0, d['dist'][-1], 5)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{'A  ' if k == 0 else 'B  ' if k == ticks[-1] else ''}{k:.1f} km\n{np.interp(k, d['dist'], d['lat']):.3f}°N\n"
                        f"{np.interp(k, d['dist'], d['lon']):.3f}°E" for k in ticks], fontsize=8)
    # 右軸：各氣壓層的平均高度
    plev = [l for l in [1000, 925, 850, 700, 500, 300, 200] if l >= SECTION['top']]
    pz = [Z[np.argmin(np.abs(d['p'][:, 0] - l))].mean() for l in plev]
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(pz)
    ax2.set_yticklabels([f'{l} hPa' for l in plev], fontsize=8)
    ax.grid(axis='y', color='gray', lw=0.3, alpha=0.5)

    # 小地圖：剖面位置
    m = fig.add_axes([0.83, 0.45, 0.16, 0.45], projection=PC)
    m.set_extent([d['lon'].min() - 0.15, d['lon'].max() + 0.15,
                  d['lat'].min() - 0.08, d['lat'].max() + 0.08], PC)
    m.add_feature(cfeature.LAND.with_scale(RES), facecolor='#f4f1e8')
    m.add_feature(cfeature.OCEAN.with_scale(RES), facecolor='#eaf3f8')
    m.add_feature(cfeature.COASTLINE.with_scale(RES), lw=0.6, edgecolor='#555')
    m.plot(d['lon'], d['lat'], color='red', lw=1.8, transform=PC)
    for name, (plo, pla) in zip('AB', SECTION['points']):
        m.plot(plo, pla, 'ko', ms=3, transform=PC)
        m.text(plo + 0.015, pla, name, fontsize=8, va='center', transform=PC)
    gl = m.gridlines(draw_labels=True, lw=0.3, color='gray', alpha=0.6, linestyle=':')
    gl.top_labels = gl.left_labels = False
    gl.xlabel_style = gl.ylabel_style = dict(size=7)

    tstr = np.datetime_as_string(t, unit='h').replace('T', ' ') + ' UTC'
    ax.set_title(f'Cross-section  |  {tstr}', fontsize=13, loc='left')
    ax.set_title('ERA Reanalysis', fontsize=10, loc='right')
    ax.annotate('Credit: S.-H. Ho', xy=(1, 1), xycoords='axes fraction', xytext=(0, 22),
                textcoords='offset points', fontsize=7, color='#444', ha='right', va='bottom')
    return ax

## 7. 互動面板
選好高度層、變數與模式後按 **繪圖 / 輸出 MP4**。MP4 每張約 2–4 秒，72 小時每小時一張約需 3–5 分鐘。

In [ ]:
import html, time, traceback
import ipywidgets as W
from IPython.display import display, Video

LEVEL_OPTS = [('地面 Surface', 'Surface')] + [(f'{int(p)} hPa', int(p)) for p in
              [1000, 925, 850, 700, 500, 300, 250, 200] if p in LEVELS] + \
             [('垂直剖面 Cross-section', 'Section')]
TIME_OPTS = [(np.datetime_as_string(t, unit='h').replace('T', ' ') + 'Z', t) for t in TIMES]
FULL, CUSTOM = '完整天氣圖 (依規格全部元素)', '自訂組合 (多選)…'


def elements_of(level):
    return ELEMENTS[level if level in ('Surface', 'Section') else 'upper']


def full_chart(level):
    """完整天氣圖預設元素：地面含鋒面；高空不含鋒面 (850 hPa 鋒面為「必要時」，可自訂加入)、不含風標"""
    if level == 'Section':
        return [e for e in elements_of(level) if not e.startswith(('相當位溫', '法向風'))]
    els = [e for e in elements_of(level) if not e.startswith('風標')]
    return els if level == 'Surface' else [e for e in els if not e.startswith('鋒面')]


w_level = W.Dropdown(options=LEVEL_OPTS, value='Surface', description='高度層')
w_proj = W.Dropdown(options=list(PROJECTIONS), value=DEFAULT_PROJ, description='投影')
w_var = W.Dropdown(description='變數')
w_multi = W.SelectMultiple(description='元素', rows=7, layout=W.Layout(width='360px'))
w_mode = W.ToggleButtons(options=[('單一時間 (PNG)', 'single'), ('時段動畫 (MP4)', 'period')],
                         description='模式')
w_time = W.Dropdown(options=TIME_OPTS, value=TIMES[0], description='時間')
w_t0 = W.Dropdown(options=TIME_OPTS, value=TIMES[0], description='開始')
w_t1 = W.Dropdown(options=TIME_OPTS, value=TIMES[-1], description='結束')
w_step = W.Dropdown(options=[('每 1 小時', 1), ('每 3 小時', 3), ('每 6 小時', 6), ('每 12 小時', 12)],
                    value=3, description='間隔')
w_fps = W.IntSlider(value=3, min=1, max=12, description='FPS')
w_btn = W.Button(description='繪圖', button_style='primary', icon='paint-brush')
w_prog = W.IntProgress(value=0, min=0, max=1, layout=W.Layout(width='360px', visibility='hidden'))
# 結果直接寫進 widget 的值，不用 W.Output：VS Code 會在每次重跑 cell 後重複顯示 Output 的內容
w_status = W.HTML()
w_img = W.Image(format='png', layout=W.Layout(width='900px', display='none'))
w_vid = W.HTML()


def on_level(_=None):
    els = elements_of(w_level.value)
    w_var.options = [FULL] + els + [CUSTOM]
    w_var.value = FULL
    w_multi.options = els
    w_multi.value = tuple(full_chart(w_level.value))
    w_proj.layout.display = 'none' if w_level.value == 'Section' else ''   # 剖面不需投影


def refresh(_=None):
    w_multi.layout.display = '' if w_var.value == CUSTOM else 'none'
    single = w_mode.value == 'single'
    w_time.layout.display = '' if single else 'none'
    period_box.layout.display = 'none' if single else ''
    w_btn.description = '繪圖' if single else '輸出 MP4'


def selected_elements():
    if w_var.value == FULL:
        return full_chart(w_level.value)
    if w_var.value == CUSTOM:
        return list(w_multi.value)
    return [w_var.value]


def show(status, png=None, video=None):
    w_status.value = status
    w_img.layout.display = 'none' if png is None else ''
    if png is not None:
        with open(png, 'rb') as f:
            w_img.value = f.read()
    w_vid.value = '' if video is None else Video(video, embed=True, width=800)._repr_html_()


_last_done = 0.0


def on_click(_):
    global _last_done
    # 繪圖期間多按的幾下會排隊，等上一張畫完才一起執行 → 忽略剛畫完就到的點擊
    if time.time() - _last_done < 0.5:
        return
    els = selected_elements()
    if not els:
        show('請至少選擇一個元素'); return
    if w_mode.value == 'period' and w_t1.value < w_t0.value:
        show('結束時間需晚於開始時間'); return
    show('繪圖中…')
    w_btn.disabled = True
    t_start = time.time()
    try:
        if w_mode.value == 'single':
            _, fn = save_png(w_time.value, w_level.value, els, proj=w_proj.value)
            show(f'已儲存: {fn}（{time.time() - t_start:.0f} 秒）', png=fn)
        else:
            w_prog.layout.visibility = 'visible'
            def prog(k, n):
                w_prog.max, w_prog.value, w_prog.description = n, k, f'{k}/{n}'
            fn = save_mp4(w_t0.value, w_t1.value, w_level.value, els, proj=w_proj.value,
                          step_h=w_step.value, fps=w_fps.value, progress=prog)
            show(f'已儲存: {fn}（{time.time() - t_start:.0f} 秒）', video=fn)
    except Exception:
        show(f'<pre style="color:#b00">{html.escape(traceback.format_exc())}</pre>')
    finally:
        w_btn.disabled = False
        _last_done = time.time()


w_level.observe(on_level, 'value')
w_var.observe(refresh, 'value')
w_mode.observe(refresh, 'value')
w_btn.on_click(on_click)
period_box = W.HBox([w_t0, w_t1, w_step, w_fps])
on_level(); refresh()

display(W.VBox([W.HBox([w_level, w_var, w_proj]), w_multi, w_mode, w_time, period_box,
                W.HBox([w_btn, w_prog]), w_status, w_img, w_vid]))